# When two English tokenizers disagree

Companion notebook to [the post on mariaa.tech](https://mariaa.tech/blog/when-tokenizers-disagree).

Two of the most common Python tokenizers — `nltk.word_tokenize` (Penn Treebank rules) and spaCy's default English tokenizer — disagree on what counts as a word, even for short English sentences. This notebook makes that disagreement concrete and lets you poke at it on your own examples.

**Estimated runtime:** ~2 minutes on a fresh Colab runtime (the install is the slow part).

## 1 · Setup

Run this cell once per session. In Colab the libraries aren't there by default — the `!pip install` line takes ~30s.

In [ ]:
# Install (Colab) — comment out if running locally with these already installed
!pip install -q nltk spacy
!python -m spacy download -q en_core_web_sm

import nltk
import spacy

nltk.download("punkt_tab", quiet=True)
nlp = spacy.load("en_core_web_sm")

print("nltk version :", nltk.__version__)
print("spaCy version:", spacy.__version__)

## 2 · The sentence

One short English sentence with the kinds of structures tokenizers tend to disagree on: contractions (`She's`, `Don't`, `it's`), a hyphenated compound (`state-of-the-art`), and end-of-clause punctuation.

In [ ]:
sentence = "She's reading the state-of-the-art paper. Don't tell her it's mine."

nltk_tokens  = nltk.word_tokenize(sentence)
spacy_tokens = [t.text for t in nlp(sentence)]

print("nltk  ({} tokens):".format(len(nltk_tokens)))
print(" ", nltk_tokens)
print()
print("spaCy ({} tokens):".format(len(spacy_tokens)))
print(" ", spacy_tokens)

## 3 · Where they disagree

Set difference on the two token lists. Anything in `In nltk only` or `In spaCy only` is a place the two libraries made different design choices.

In [ ]:
nltk_set  = set(nltk_tokens)
spacy_set = set(spacy_tokens)

only_nltk  = sorted(nltk_set - spacy_set)
only_spacy = sorted(spacy_set - nltk_set)
shared     = sorted(nltk_set & spacy_set)

print("In nltk only :", only_nltk)
print("In spaCy only:", only_spacy)
print()
print("Shared       :", shared)

## 4 · Side-by-side

Walk the two token lists in parallel. Where the columns differ, that's the disagreement.

In [ ]:
from itertools import zip_longest

header = f"{'#':>3}  {'nltk':<22} {'spaCy':<22}  match?"
print(header)
print("-" * len(header))

for i, (a, b) in enumerate(zip_longest(nltk_tokens, spacy_tokens, fillvalue="—"), start=1):
    flag = "" if a == b else "  <-- diff"
    print(f"{i:>3}  {a!r:<22} {b!r:<22}{flag}")

## 5 · Why they disagree (the short version)

Both tokenizers are *correct*. They answer different questions:

- **nltk's `word_tokenize`** follows the **Penn Treebank** corpus convention from the 1990s. Deterministic, well-documented, conservative.
- **spaCy's default English tokenizer** is a **rule pipeline of prefixes, suffixes, infixes, and special cases**, tuned for downstream model quality. Less predictable across versions, more useful in modern pipelines.

The pipeline-level lesson: **pick the tokenizer that matches what comes next**, not the one you've heard of. For transformer models (BERT, RoBERTa, GPT-style), use the model's *own* tokenizer — BPE, WordPiece, or SentencePiece — not either of these.

## 6 · Your turn

Try a sentence with the structures that break tokenizers: contractions, possessives, hyphens, URLs, decimals, emoji, hashtags. Edit the `test` string and re-run.

In [ ]:
test = "Maria's blog at mariaa.tech is state-of-the-art. Don't @ me."

print("nltk :", nltk.word_tokenize(test))
print("spaCy:", [t.text for t in nlp(test)])

## 7 · Tokenizing a non-space language

Hand any of these to a whitespace splitter and you get the whole sentence back as one token. Real tokenization here requires a *learned* model — not a rule-based splitter. spaCy supports it via language-specific pipelines (`ja_core_news_sm` for Japanese, etc.). This is the *real* difficulty of layer 1.

In [ ]:
japanese = "犬が男の子を追いかけている。"
chinese  = "狗在操场上追男孩。"
thai     = "สุนัขกำลังไล่ตามเด็กผู้ชาย"

for label, s in [("Japanese", japanese), ("Chinese", chinese), ("Thai", thai)]:
    print(f"{label:>9}: {s.split()}  ← whitespace split, useless")